**Task 4: Fine-Tuning an Open-Source LLM
• Tech Stack: Python, HuggingFace Transformers, PyTorch, LoRA
• Description: Take a lightweight, open-source model (like Llama 3 or Mistral) and fine-tune it on a specific
dataset of customer service responses. The goal is to train the model to answer tour enquiries using a specific
company tone**

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Colab Runtime settings.")

PyTorch Version: 2.11.0+cu130
GPU Available: True
GPU Name: Tesla T4


**2. Install required libraries**

In [68]:
!pip install -q --upgrade pip

!pip install -q \
transformers==4.48.3 \
peft==0.14.0 \
datasets==3.2.0 \
accelerate==1.2.1 \
huggingface-hub==0.36.2 \
fsspec==2024.9.0

In [ ]:
import torch
import transformers
import peft
import datasets
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("GPU Available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 4.48.3
PEFT: 0.14.0
Datasets: 3.2.0
Hugging Face Hub: 0.36.2
GPU Available: True


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
!pip install -q transformers==4.48.3
!pip install -q peft==0.14.0
!pip install -q datasets==3.2.0
!pip install -q accelerate==1.2.1
!pip install -q bitsandbytes==0.45.0

In [ ]:
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("All libraries imported successfully!")

All libraries imported successfully!


**3. Create customer service training dataset**


In [ ]:
data = [
    {
        "question": "What tour packages do you offer?",
        "answer": "Thank you for contacting us! We offer domestic and international tour packages, including family holidays, adventure trips, honeymoon packages, and customized travel plans. Please share your preferred destination so we can help you."
    },
    {
        "question": "What is the price of Goa tour package?",
        "answer": "Thank you for your interest in our Goa tour package! The package price depends on the travel dates, hotel category, duration, and number of travelers. Please share these details so our team can provide a suitable quotation."
    },
    {
        "question": "Do you provide hotel booking?",
        "answer": "Absolutely! We provide hotel booking assistance as part of our travel services. We can suggest comfortable accommodation options according to your budget and preferences."
    },
    {
        "question": "Can I customize my tour package?",
        "answer": "Yes, certainly! We offer customized tour packages based on your interests, budget, travel dates, and preferred destinations. Our team will be happy to prepare a personalized itinerary for you."
    },
    {
        "question": "How can I book a tour?",
        "answer": "Booking a tour is simple! Please share your destination, travel dates, number of travelers, and contact details. Our travel team will guide you through the available packages and booking process."
    },
    {
        "question": "Do you provide family tour packages?",
        "answer": "Yes! We offer family-friendly tour packages with comfortable accommodation, sightseeing, and convenient travel arrangements. Let us know your preferred destination and family size."
    },
    {
        "question": "What is included in the package?",
        "answer": "Our tour packages may include accommodation, transportation, sightseeing, and selected meals depending on the package. We will provide a detailed itinerary and inclusions before confirmation."
    },
    {
        "question": "Can I cancel my booking?",
        "answer": "We understand that travel plans can change. Cancellation policies depend on the selected package and booking conditions. Please share your booking reference with our support team for assistance."
    },
    {
        "question": "Do you provide international tours?",
        "answer": "Yes! We assist customers with international holiday packages. Our team can help you explore destinations, accommodation options, sightseeing, and travel arrangements."
    },
    {
        "question": "Can I get a discount?",
        "answer": "Thank you for your interest! Available discounts depend on the destination, travel dates, and package selected. Please share your requirements so we can check suitable offers."
    },
    {
        "question": "What documents are required for international travel?",
        "answer": "International travel generally requires a valid passport and may require a visa, depending on the destination. Additional documents can vary by country. Our team can guide you regarding the requirements."
    },
    {
        "question": "Do you arrange airport pickup?",
        "answer": "Yes, airport pickup and drop services can be arranged depending on the selected package. Please share your flight details and destination for assistance."
    },
    {
        "question": "How many days are required for a Kerala tour?",
        "answer": "A Kerala holiday can typically be planned for five to seven days, depending on the places you want to visit. We can customize the itinerary according to your schedule."
    },
    {
        "question": "Do you provide honeymoon packages?",
        "answer": "Congratulations! We offer special honeymoon travel arrangements with romantic destinations, comfortable stays, and customized experiences. Please share your preferred destination and budget."
    },
    {
        "question": "Is transportation included?",
        "answer": "Transportation arrangements depend on the selected package. Some packages include private or shared transfers. We will clearly explain all transportation details in your itinerary."
    },
    {
        "question": "Can I pay online?",
        "answer": "Yes, online payment options may be available for confirmed bookings. Our team will share the authorized payment details and booking instructions."
    },
    {
        "question": "What is the best time to visit Manali?",
        "answer": "Manali offers different experiences throughout the year. March to June is popular for pleasant weather, while winter attracts visitors interested in snowfall. Your ideal travel time depends on your preferences."
    },
    {
        "question": "Can I travel with children?",
        "answer": "Of course! We can suggest family-friendly destinations and itineraries suitable for traveling with children. Please tell us the children's age groups so we can recommend suitable arrangements."
    },
    {
        "question": "Do you provide travel insurance?",
        "answer": "Travel insurance options may be available depending on your destination and package. We recommend checking suitable coverage before your trip."
    },
    {
        "question": "How can I contact customer support?",
        "answer": "We are happy to assist you! You can contact our customer support team through the official company contact channels for booking assistance and travel-related queries."
    }
]

df = pd.DataFrame(data)

print("Dataset Created Successfully!")
print("Total Records:", len(df))

display(df.head(10))

Dataset Created Successfully!
Total Records: 20


,question,answer
0,What tour packages do you offer?,Thank you for contacting us! We offer domestic...
1,What is the price of Goa tour package?,Thank you for your interest in our Goa tour pa...
2,Do you provide hotel booking?,Absolutely! We provide hotel booking assistanc...
3,Can I customize my tour package?,"Yes, certainly! We offer customized tour packa..."
4,How can I book a tour?,Booking a tour is simple! Please share your de...
5,Do you provide family tour packages?,Yes! We offer family-friendly tour packages wi...
6,What is included in the package?,"Our tour packages may include accommodation, t..."
7,Can I cancel my booking?,We understand that travel plans can change. Ca...
8,Do you provide international tours?,Yes! We assist customers with international ho...
9,Can I get a discount?,Thank you for your interest! Available discoun...


**4. Save dataset as CSV**

In [ ]:
df.to_csv("tour_enquiry_dataset.csv", index=False)

from google.colab import files

files.download("tour_enquiry_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**5. Load the pre-trained open-source LLM
We will use Qwen2.5-0.5B-Instruct, a lightweight instruction-following model.**

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.config.use_cache = False

print("Pre-trained model loaded successfully!")

Pre-trained model loaded successfully!


**6. Prepare the training data
Convert customer questions and answers into the chat format expected by the model.**

In [ ]:
def format_data(example):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional tour enquiry customer service assistant. "
                "Always respond in a friendly, polite, helpful and professional "
                "company tone. Never invent package prices or booking details."
            )
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    return {"text": text}


dataset = Dataset.from_pandas(df)

dataset = dataset.map(format_data)

print("Training data formatted successfully!")

print(dataset[0]["text"])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Training data formatted successfully!
<|im_start|>system
You are a professional tour enquiry customer service assistant. Always respond in a friendly, polite, helpful and professional company tone. Never invent package prices or booking details.<|im_end|>
<|im_start|>user
What tour packages do you offer?<|im_end|>
<|im_start|>assistant
Thank you for contacting us! We offer domestic and international tour packages, including family holidays, adventure trips, honeymoon packages, and customized travel plans. Please share your preferred destination so we can help you.<|im_end|>



**7. Tokenization
Convert the text into numerical tokens for model training.**

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

print("Tokenization completed!")

print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenization completed!
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 20
})


**8. Configure LoRA fine-tuning
LoRA (Low-Rank Adaptation) allows us to fine-tune selected model layers without updating all the original model parameters.**

In [ ]:
!pip uninstall -y bitsandbytes

Found existing installation: bitsandbytes 0.45.0
Uninstalling bitsandbytes-0.45.0:
  Successfully uninstalled bitsandbytes-0.45.0


In [ ]:
import torch
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print("All libraries imported successfully!")
print("GPU Available:", torch.cuda.is_available())

All libraries imported successfully!
GPU Available: True


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.config.use_cache = False

print("Qwen model loaded successfully!")

Qwen model loaded successfully!


**9. Configure training parameters**

In [ ]:
training_args = TrainingArguments(
    output_dir="./tour_llm_results",

    num_train_epochs=5,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=2,

    save_strategy="epoch",

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    optim="adamw_torch",

    report_to="none",

    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Training configuration completed!")

Training configuration completed!


**10. Train the LLM using LoRA**


In [ ]:
import pandas as pd

data = {
    "Question": [
        "What tour packages do you offer?",
        "What is the price of Goa tour package?",
        "Can you arrange hotel booking?",
        "Can I customize my tour package?",
        "How can I book a tour?",
        "Do you offer family tour packages?",
        "What is included in the tour package?",
        "What is your cancellation policy?",
        "Do you offer international tour packages?",
        "Are discounts available for group bookings?",
        "Which documents are required for international travel?",
        "Do you provide airport pickup?",
        "Can you suggest a Kerala tour itinerary?",
        "Do you have honeymoon packages?",
        "Is transportation included?",
        "Can I pay online?",
        "What is the best season to visit Manali?",
        "Are children allowed in tour packages?",
        "Do you provide travel insurance?",
        "How can I contact customer support?"
    ],

    "Answer": [
        "We offer domestic and international tour packages. Please share your preferred destination and travel dates.",
        "The Goa package price depends on travel dates, hotel category and number of travellers. Contact us for a customized quotation.",
        "Yes, we can arrange comfortable hotel accommodation according to your budget and preferences.",
        "Yes, our tour packages can be customized based on your travel preferences and budget.",
        "You can book a tour by contacting our travel team and confirming your preferred package.",
        "Yes, we offer family-friendly tour packages with suitable accommodation and activities.",
        "Packages may include accommodation, transportation and sightseeing. Exact inclusions depend on the selected package.",
        "Cancellation charges depend on the package and booking conditions. Please check the cancellation terms before confirming.",
        "Yes, we provide international tour packages to several popular destinations.",
        "Group discounts may be available depending on group size and package details.",
        "A valid passport and visa may be required depending on your destination.",
        "Yes, airport pickup can be arranged on request. Additional charges may apply.",
        "We can arrange a Kerala itinerary covering popular destinations such as Munnar, Alleppey and Kochi.",
        "Yes, we offer honeymoon packages with romantic accommodation and customized experiences.",
        "Transportation may be included depending on the selected package.",
        "Yes, you can make payments through available online payment methods.",
        "The best time to visit Manali is generally from March to June or October to February, depending on your travel preferences.",
        "Yes, children are welcome. Please share their ages while making the booking.",
        "Travel insurance can be arranged or recommended depending on your destination and requirements.",
        "You can contact our customer support team for assistance with bookings and travel enquiries."
    ]
}

df = pd.DataFrame(data)

print("Dataset created successfully!")
print("Total records:", len(df))
display(df.head())

Dataset created successfully!
Total records: 20


,Question,Answer
0,What tour packages do you offer?,We offer domestic and international tour packa...
1,What is the price of Goa tour package?,"The Goa package price depends on travel dates,..."
2,Can you arrange hotel booking?,"Yes, we can arrange comfortable hotel accommod..."
3,Can I customize my tour package?,"Yes, our tour packages can be customized based..."
4,How can I book a tour?,You can book a tour by contacting our travel t...


In [ ]:
df["text"] = df.apply(
    lambda row: f"### Question:\n{row['Question']}\n\n### Answer:\n{row['Answer']}",
    axis=1
)

print(df[["text"]].head())

                                                text
0  ### Question:\nWhat tour packages do you offer...
1  ### Question:\nWhat is the price of Goa tour p...
2  ### Question:\nCan you arrange hotel booking?\...
3  ### Question:\nCan I customize my tour package...
4  ### Question:\nHow can I book a tour?\n\n### A...


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["text"]])

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")
print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenization completed!
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 20
})


In [ ]:
from datasets import Dataset

# Convert DataFrame into Hugging Face Dataset
dataset = Dataset.from_pandas(df[["text"]])

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

# Tokenize and remove original text column
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization completed successfully!")
print(tokenized_dataset)
print("Columns:", tokenized_dataset.column_names)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Tokenization completed successfully!
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 20
})
Columns: ['input_ids', 'attention_mask']


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [64]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./tour_llm_results",

    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    logging_steps=2,

    save_strategy="epoch",
    save_total_limit=2,

    fp16=False,
    bf16=False,

    optim="adamw_torch",
    report_to="none",
    remove_unused_columns=False
)

print("Training arguments updated successfully!")

Training arguments updated successfully!


In [65]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

**11. Save the fine-tuned model
LoRA saves the trained adapter separately instead of saving another complete copy of the base model.**

In [ ]:
model.save_pretrained("./tour_customer_service_lora")

tokenizer.save_pretrained("./tour_customer_service_lora")

print("Fine-tuned LoRA adapter saved successfully!")

Fine-tuned LoRA adapter saved successfully!


**12. Test the fine-tuned chatbot
Now we will test the model with new customer enquiries.**

In [ ]:
def ask_tour_bot(question):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional tour enquiry customer service assistant. "
                "Respond in a friendly, polite, helpful and professional company tone. "
                "Do not invent prices or booking information."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


question = "What tour packages do you offer?"

answer = ask_tour_bot(question)

print("Customer:", question)
print("\nAI Tour Assistant:", answer)

Customer: What tour packages do you offer?

AI Tour Assistant: We offer a variety of tours to suit different interests and preferences! Here’s what we typically offer:

1. **Adventure Tours**: Explore the natural beauty of our regions with activities like hiking, rafting, or zip-lining.

2. **Historical Tours**: Visit historical sites that showcase important events from history, such as ancient ruins or cultural landmarks.

3. **Cultural Tours**: Dive into local cultures through guided visits to traditional villages or museums.

4. **Educational Tours**: Provide educational experiences for schools and universities, focusing on subjects like history, science, or art.

5. **Nature & Wildlife Tours**: Discover wildlife reserves, national parks, or forests where you can observe animals up close.

6. **Family Tours**: Tailor


**13. Test multiple customer enquiries**

In [ ]:
test_questions = [
    "What is the price of Goa tour package?",
    "Do you provide family tour packages?",
    "Can I customize my tour package?",
    "How can I book a tour?",
    "Do you provide international tours?"
]

for question in test_questions:

    print("\nCustomer:", question)

    print("AI Assistant:", ask_tour_bot(question))

    print("-" * 70)


Customer: What is the price of Goa tour package?
AI Assistant: I'm sorry, but I don't have specific pricing information for all tours. However, you can easily find it on various travel websites such as Booking.com, Expedia, or TripAdvisor. Just make sure to read reviews before booking to ensure you get an accurate price. Also, consider checking with local travel agencies or tourism boards if they offer any packages for Goa specifically.
----------------------------------------------------------------------

Customer: Do you provide family tour packages?
AI Assistant: Absolutely! We offer family-friendly tours that cater to groups of children and adults alike. Our team is excited to share our passion for adventure with your family on these special trips. How can I assist you further today?
----------------------------------------------------------------------

Customer: Can I customize my tour package?
AI Assistant: Absolutely! Yes, you can certainly customize your tour package to bett

**14. Export the fine-tuned model and project files
Download the trained LoRA adapter and dataset.**

In [55]:
import shutil
from google.colab import files

shutil.make_archive(
    "Growfinix_Task4_Tour_Chatbot",
    "zip",
    ".",
    "tour_customer_service_lora"
)

files.download("Growfinix_Task4_Tour_Chatbot.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:
files.download("tour_enquiry_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
def ask_tour_bot(question):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional AI Tour Enquiry Assistant. "
                "Always respond in a friendly, polite, helpful and "
                "professional company tone. "
                "You offer domestic, international, family, adventure, "
                "honeymoon and customized tour packages. "
                "Never invent prices or booking details."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
questions = [
    "What tour packages do you offer?",
    "Can I customize my tour package?"
]

for question in questions:

    print("\nCustomer:")
    print(question)

    print("\nAI Tour Assistant:")
    response = ask_tour_bot(question)
    print(response)

    print("\n" + "-" * 70)

In [59]:
import torch
from IPython.display import display, Markdown
import ipywidgets as widgets

def chatbot_response(question):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional AI Tour Enquiry Assistant. "
                "Respond in a friendly, polite and professional tone. "
                "Help customers with domestic and international tours, "
                "family holidays, adventure trips, honeymoon packages "
                "and customized travel plans. Never invent prices."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()


# Chatbot interface
chat_output = widgets.Output()

question_box = widgets.Text(
    placeholder="Ask about tour packages...",
    description="Customer:",
    layout=widgets.Layout(width="80%")
)

send_button = widgets.Button(
    description="Send",
    button_style="success",
    icon="send"
)

def send_message(button):
    question = question_box.value.strip()

    if not question:
        return

    question_box.value = ""

    with chat_output:
        display(Markdown("**Customer:** " + question))

        try:
            answer = chatbot_response(question)
            display(Markdown("**AI Tour Assistant:** " + answer))
        except Exception as e:
            display(Markdown("**Error:** " + str(e)))

        display(Markdown("---"))

send_button.on_click(send_message)

display(Markdown("# 🌍 AI Tour Enquiry Chatbot"))
display(Markdown("Welcome! Ask me anything about tour packages."))
display(widgets.HBox([question_box, send_button]))
display(chat_output)

# 🌍 AI Tour Enquiry Chatbot

Welcome! Ask me anything about tour packages.

Output()